In [4]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
import warnings

from Bio import BiopythonWarning
from Bio.PDB import PDBIO, Select
warnings.simplefilter("ignore", BiopythonWarning)

from placer.process import structure as structure

class ProteinWithLigand(Select):
    def __init__(self, keep_lig):
        self.keep_lig = keep_lig  # e.g. "ATP" or ["ATP", "AMP"]
        if isinstance(self.keep_lig, str):
            self.keep_lig = [self.keep_lig]

    def accept_residue(self, residue):
        # keep standard amino acids
        if residue.id[0] == " ":
            return True
        # keep specified ligands
        if residue.get_resname().strip() in self.keep_lig:
            return True
        return False

In [5]:
import os
import shutil

src_dir = "outputs/"
dst_dir = "/home/hgji/anaconda3/mbel/tools/AutoDock-Vina/targets/maCAR_rational/pdb_files"

os.makedirs(dst_dir, exist_ok=True)

for fname in os.listdir(src_dir):
    if fname.endswith(".pdb"):
        pdb_path = os.path.join(src_dir, fname)
        models = structure.load_models_from_pdb(pdb_path)
        rep_model = models[0]
        
        io = PDBIO()
        io.set_structure(rep_model)
        io.save(os.path.join(dst_dir, fname), ProteinWithLigand("ATP"))
